# Section 2: NumPy-Only Image Classifier

**Goal:** Build a small image classifier using **only NumPy**. You will reuse functions from **Section 1** (forward, loss, backward, update), write a **tiny CNN model** (layers + forward/backward), tune **hyperparameters**, and finally **report test accuracy**.

---

## What you need do

1. **Copy your Task 2 functions** into this notebook (we provide empty placeholders). You should paste your implementations to replace the placeholders.

2. **Write your model**: Implement a small CNN for classification (e.g., `Conv → ReLU → Pool → Conv → ReLU → Pool → FC → ReLU → FC`). We provide **one example layer** and fixed input/output shapes to illustrate I/O.

3. **Train & Evaluate**: Run the training loop we provide (uses `tqdm` and timing), then output **validation accuracy** as the grading metric.

> Dataset folder layout must be:
```
dataset/
├── train/
│   ├── class0/
│   ├── class1/
│   └── ...
└── val/
    ├── class0/
    ├── class1/
    └── ...
```
Images will be loaded as **grayscale** and resized to **100×100** by default.


In [1]:
# === Imports & Global Config ===
import os, time, random
from typing import List, Tuple
import numpy as np
from PIL import Image
from tqdm import tqdm
import json
from google.colab import drive
# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Default image size (matches your dataset)
IMG_SIZE = 100


## Part A — Copy your **Section 1** functions here

**IMPORTANT:** Replace the bodies of the following functions with your **correct** implementations from Section 1.  

If you leave them as-is, the notebook will raise `NotImplementedError` and you will not be able to train.


In [2]:
# ===== Task 1: Linear layer (forward) =====

    # TODO: paste your solution
def linear_layer(x, W, theta):
    """
    Linear (fully connected) layer forward pass.

    Parameters:
      x      : (N, d_in)   input matrix
      W      : (d_in, d_out) weight matrix
      theta  : (d_out,)    bias vector

    Returns:
      y : (N, d_out)       output of this layer
    """

    # Write your code here
    y = x @ W + theta

    return y


In [3]:
# ===== Task 2: 2D Convolution (single-channel, valid, stride=1, no bias) =====
    # TODO: paste your solution
def conv2d(X: np.ndarray,
           kernel: np.ndarray) -> np.ndarray:
    """
    2D convolution (single-channel, no bias, no padding, stride = 1)

    X       : (H, W)          input image
    kernel  : (k, k)          filter weights

    returns : (H_out, W_out)  output feature map
    """
    assert X.ndim == 2, "X must be 2D with shape (H, W)"
    assert kernel.ndim == 2 and kernel.shape[0] == kernel.shape[1], "kernel must be square (k x k)"

    H, W = X.shape
    k = kernel.shape[0]

    # Output size (valid convolution: only where the kernel fully fits inside the input)
    H_out = H - k + 1
    W_out = W - k + 1

    Y = np.zeros((H_out, W_out), dtype=np.float32)

    # Convolution: slide 1 pixel at a time and store results in output matrix Y
    for i in range(H_out):
        for j in range(W_out):
            window = X[i:i+k, j:j+k]
            Y[i, j] = np.sum(window * kernel)

    return Y


In [4]:
# ===== Task 3: ReLU =====
    # TODO: paste your solution
def ReLU(X: np.ndarray) -> np.ndarray:
    """
    ReLU activation: max(0, x)
    """
    output_matrix = np.maximum(0, X)
    return output_matrix


In [5]:
# ===== Task 4: Max Pool 2D (k=2) =====
    # TODO: paste your solution
def max_pool2d(X: np.ndarray, k: int = 2) -> np.ndarray:
    """
    2D max pooling for teaching use.
    X      : (H, W)
    k      : window size

    returns: (H_out, W_out)
    """
    assert X.ndim == 2, "X must be 2D"
    H, W = X.shape
    H_out = H // k
    W_out = W // k

    Y = np.zeros((H_out, W_out), dtype=np.float32)
    for i in range(H_out):
        for j in range(W_out):
            window = X[i*k:(i+1)*k, j*k:(j+1)*k]
            Y[i, j] = np.max(window)

    return Y


In [6]:
# ===== Task 5: MSE loss (value and dY) =====
    # TODO: paste your solution
def mse_loss(y_pred: np.ndarray, y_true: np.ndarray):
    """
    Compute Mean Squared Error and its gradient by y_pred.

    y_pred : (N, d_out)
    y_true : (N, d_out)

    Returns:
      loss : mse_loss result
      dY   : gradient of loss w.r.t. y_pred, same shape as y_pred
    """

    # Write your code here

    diff = y_pred - y_true
    loss = np.mean(diff ** 2)
    n = y_pred.size
    dY = (2.0 / n) * diff


    return loss, dY


In [7]:
# ===== Linear backward for y = xW + theta =====
def backward(dY: np.ndarray, x: np.ndarray, W: np.ndarray):
    # TODO: paste your solution
    dW = np.dot(x.T, dY)                # shape (d_in, d_out)
    dTheta = np.sum(dY, axis=0)         # shape (d_out,)
    dX = np.dot(dY, W.T)                # shape (N, d_in)
    return dW, dTheta, dX


In [8]:
# ===== Task 6: Parameter update (GD) =====
    # TODO: paste your solution
def update_params(W, theta, dW, dTheta, lr=0.01):
    """
    Gradient Descent parameter update.
    Update Parameters W and theta for y = xW + theta
    W : weight matrix
    theta : bias vector
    dW : gradient of loss w.r.t W
    dTheta : gradient of loss w.r.t bias
    lr : learning rate
    Returns updated W and theta.
    """
    W_new = W - lr * dW
    theta_new = theta - lr * dTheta
    return W_new, theta_new

## Part B — Provided utilities and tools (Do not modify)

We provide **softmax**, **cross-entropy**, **accuracy**, and a **dataset loader** for you.


In [9]:
def relu_backward(dY, X):
    return dY * (X > 0)

def pad2d_zero(X, pad_top, pad_bottom, pad_left, pad_right):
    """Zero-pad a 2D array."""
    H, W = X.shape
    Y = np.zeros((H + pad_top + pad_bottom, W + pad_left + pad_right), dtype=X.dtype)
    Y[pad_top:pad_top+H, pad_left:pad_left+W] = X
    return Y

def conv2d_multi_forward_basic(XN, K):
    """
    Forward pass for multiple filters applied to a single-channel input.
    Uses only the basic conv2d().

    XN: (N, H, W)     batch of single-channel images
    K:  (F, k, k)     filters
    Returns:
        Y: (N, H-k+1, W-k+1, F)
        cache for backward
    """
    N, H, W = XN.shape
    F, k, _ = K.shape
    H_out, W_out = H - k + 1, W - k + 1
    Y = np.zeros((N, H_out, W_out, F), dtype=np.float32)
    for n in range(N):
        for f in range(F):
            Y[n, :, :, f] = conv2d(XN[n], K[f])
    cache = (XN, K)
    return Y, cache


def conv2d_multi_backward_basic(dY, cache):
    """
    Backward pass for conv2d_multi_forward_basic, using only conv2d().
    dK[f] = sum_n conv2d(XN[n], dY[n, :, :, f])
    dX[n] = sum_f conv2d( padded(dY[n, :, :, f]), rot180(K[f]) )

    dY: (N, H_out, W_out, F)
    Returns:
        dX: (N, H, W)
        dK: (F, k, k)
    """
    XN, K = cache
    N, H, W = XN.shape
    F, k, _ = K.shape
    H_out, W_out, _ = dY.shape[1:]

    # gradient w.r.t. filters
    dK = np.zeros_like(K, dtype=np.float32)
    for f in range(F):
        for n in range(N):
            dK[f] += conv2d(XN[n], dY[n, :, :, f])

    # gradient w.r.t. input (full convolution)
    dX = np.zeros_like(XN, dtype=np.float32)
    for n in range(N):
        for f in range(F):
            kflip = np.flip(np.flip(K[f], axis=0), axis=1)
            pad = k - 1
            dYpad = pad2d_zero(dY[n, :, :, f], pad, pad, pad, pad)
            dX[n] += conv2d(dYpad, kflip)
    return dX, dK

def maxpool2d_forward_basic(XN, k=2):
    """
    Forward pass of max-pooling using the basic max_pool2d.
    Builds an argmax mask for backward.

    XN: (N, H, W, C) or (N, H, W)
    Returns:
        Y: pooled output
        mask: boolean mask for backward
    """
    if XN.ndim == 3:
        XN = XN[..., None]
    N, H, W, C = XN.shape
    H_out, W_out = H // k, W // k
    Y = np.zeros((N, H_out, W_out, C), dtype=np.float32)
    mask = np.zeros_like(XN, dtype=bool)
    for n in range(N):
        for c in range(C):
            Y[n, :, :, c] = max_pool2d(XN[n, :, :, c], k=k)
            for i in range(H_out):
                for j in range(W_out):
                    window = XN[n, i*k:(i+1)*k, j*k:(j+1)*k, c]
                    idx = np.unravel_index(np.argmax(window), (k, k))
                    mask[n, i*k+idx[0], j*k+idx[1], c] = True
    return Y, mask


def maxpool2d_backward_basic(dY, mask, k=2):
    """Backward pass for max-pooling using the saved mask."""
    if dY.ndim == 3:
        dY = dY[..., None]
    N, H, W, C = mask.shape
    H_out, W_out = dY.shape[1:3]
    dX = np.zeros(mask.shape, dtype=np.float32)
    for n in range(N):
        for c in range(C):
            for i in range(H_out):
                for j in range(W_out):
                    window_mask = mask[n, i*k:(i+1)*k, j*k:(j+1)*k, c]
                    dX[n, i*k:(i+1)*k, j*k:(j+1)*k, c][window_mask] = dY[n, i, j, c]
    if dX.shape[-1] == 1:
        dX = dX[..., 0]
    return dX

def conv2d_mimo_forward_basic(XNHWC, K):
    """
    Multi-input, multi-output convolution using only basic conv2d.

    XNHWC: (N, H, W, C_in)
    K:      (C_out, k, k, C_in)
    Returns:
        Y: (N, H-k+1, W-k+1, C_out)
        cache for backward
    """
    if XNHWC.ndim == 3:
        XNHWC = XNHWC[..., None]
    N, H, W, C_in = XNHWC.shape
    C_out, k, _, CinK = K.shape
    assert CinK == C_in
    H_out, W_out = H - k + 1, W - k + 1
    Y = np.zeros((N, H_out, W_out, C_out), dtype=np.float32)
    for n in range(N):
        for co in range(C_out):
            acc = np.zeros((H_out, W_out), dtype=np.float32)
            for ci in range(C_in):
                acc += conv2d(XNHWC[n, :, :, ci], K[co, :, :, ci])
            Y[n, :, :, co] = acc
    cache = (XNHWC, K)
    return Y, cache


def conv2d_mimo_backward_basic(dY, cache):
    """
    Backward for multi-input multi-output conv, using only conv2d.
    For each output channel co:
        dK[co, :, :, ci] = sum_n conv2d(X[n, ci], dY[n, co])
        dX[n, :, :, ci] += conv2d(padded(dY[n, co]), rot180(K[co, :, :, ci]))
    """
    X, K = cache
    if X.ndim == 3:
        X = X[..., None]
    N, H, W, C_in = X.shape
    C_out, k, _, CinK = K.shape
    assert CinK == C_in
    H_out, W_out, CoutDY = dY.shape[1:]
    assert CoutDY == C_out

    dK = np.zeros_like(K, dtype=np.float32)
    dX = np.zeros_like(X, dtype=np.float32)

    # dK
    for co in range(C_out):
        for ci in range(C_in):
            for n in range(N):
                dK[co, :, :, ci] += conv2d(X[n, :, :, ci], dY[n, :, :, co])

    # dX
    pad = k - 1
    for n in range(N):
        for co in range(C_out):
            dYpad = pad2d_zero(dY[n, :, :, co], pad, pad, pad, pad)
            for ci in range(C_in):
                kflip = np.flip(np.flip(K[co, :, :, ci], axis=0), axis=1)
                dX[n, :, :, ci] += conv2d(dYpad, kflip)

    if dX.shape[-1] == 1:
        dX = dX[..., 0]
    return dX, dK

def softmax(z: np.ndarray) -> np.ndarray:
    z = z - z.max(axis=1, keepdims=True)
    ez = np.exp(z, dtype=np.float32)
    return ez / (np.sum(ez, axis=1, keepdims=True) + 1e-12)

def cross_entropy_loss(probs: np.ndarray, y_true_idx: np.ndarray):
    """
    probs: (N, C) after softmax
    y_true_idx: (N,) integer labels
    returns (loss, grad_logits)
    """
    N = probs.shape[0]
    logp = -np.log(probs[np.arange(N), y_true_idx] + 1e-12)
    loss = float(np.mean(logp))
    grad = probs.copy()
    grad[np.arange(N), y_true_idx] -= 1.0
    grad /= N
    return loss, grad

def accuracy_from_logits(logits: np.ndarray, y_true_idx: np.ndarray) -> float:
    pred = np.argmax(softmax(logits), axis=1)
    return float(np.mean(pred == y_true_idx))

In [10]:
# === Simple dataset loader for folder structure ===
def list_images(root: str) -> Tuple[list, list]:
    classes = sorted([d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))])
    items = []
    for ci, cname in enumerate(classes):
        cdir = os.path.join(root, cname)
        for fn in os.listdir(cdir):
            if fn.lower().endswith(('.jpg', '.jpeg', '.png', '.bmp')):
                items.append((os.path.join(cdir, fn), ci))
    return items, classes

def load_batch(paths_labels, start, end, img_size=IMG_SIZE, augment=False):
    batch = paths_labels[start:end]
    N = len(batch)
    X = np.zeros((N, img_size, img_size), dtype=np.float32)
    y = np.zeros((N,), dtype=np.int64)
    for i, (p, lab) in enumerate(batch):
        img = Image.open(p).convert('L')  # grayscale
        if augment:
            # basic augmentation examples
            if random.random() < 0.5:
                img = img.transpose(Image.FLIP_LEFT_RIGHT)
            if random.random() < 0.3:
                angle = random.uniform(-10, 10)
                img = img.rotate(angle, resample=Image.BILINEAR, fillcolor=0)
        img = img.resize((img_size, img_size), Image.BILINEAR)
        X[i] = np.asarray(img, dtype=np.float32) / 255.0
        y[i] = lab
    return X, y

In [11]:
def state_dict(model):
    """
    Collect all top-level numpy array attributes from the model.
    Returns a dict: {name: np.ndarray}
    """
    sd = {}
    for name, val in vars(model).items():
        if isinstance(val, np.ndarray):
            sd[name] = val
    return sd

def load_state_dict(model, state, strict=True):
    """
    Load arrays from `state` (a dict of name->ndarray) into model attributes.
    If strict=True, raises on missing keys or shape mismatch.
    Returns a report dict.
    """
    report = {"loaded": [], "missing_in_model": [], "missing_in_state": [], "shape_mismatch": []}
    model_vars = vars(model)
    # apply
    for k, arr in state.items():
        if k not in model_vars:
            report["missing_in_model"].append(k)
            if strict:
                raise KeyError(f"Parameter '{k}' not found in model.")
            continue
        if not isinstance(model_vars[k], np.ndarray):
            report["shape_mismatch"].append((k, "not an ndarray on model"))
            if strict:
                raise TypeError(f"Model attribute '{k}' is not an ndarray.")
            continue
        if model_vars[k].shape != arr.shape:
            report["shape_mismatch"].append((k, (model_vars[k].shape, arr.shape)))
            if strict:
                raise ValueError(f"Shape mismatch for '{k}': model {model_vars[k].shape} vs state {arr.shape}")
            # non-strict: still load
        setattr(model, k, arr.astype(model_vars[k].dtype, copy=True))
        report["loaded"].append(k)
    # find params that were expected but not provided
    for k, v in model_vars.items():
        if isinstance(v, np.ndarray) and k not in state:
            report["missing_in_state"].append(k)
    return report

def save_model(model, classes, path="checkpoints/model_final.npz", extra_meta=None):
    """
    Save model parameters (all top-level ndarrays) and metadata into a .npz file.
    """
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    sd = state_dict(model)
    meta = {
        "model_class": model.__class__.__name__,
        "img_size": getattr(model, "img_size", None),
        "num_classes": getattr(model, "num_classes", None),
        "lr": getattr(model, "lr", None),
        "timestamp": time.strftime("%Y-%m-%d %H:%M:%S"),
        "classes": classes,
    }
    if extra_meta:
        meta.update(extra_meta)
    # pack: all arrays under their own keys + one 'meta' json
    np.savez(path, **sd, meta=np.array([json.dumps(meta)], dtype=object))
    print(f"💾 Saved {len(sd)} tensors to {path}")
    return {"num_tensors": len(sd), "path": path, "meta": meta}

def load_model(path, model=None, model_ctor=None, ctor_kwargs=None, strict=True):
    """
    Load a saved .npz. If `model` is None, instantiate via `model_ctor(**ctor_kwargs)`.
    Returns (model, meta, report).
    """
    data = np.load(path, allow_pickle=True)
    # extract meta if present
    meta = {}
    if "meta" in data.files:
        try:
            meta = json.loads(str(data["meta"][0]))
        except Exception:
            pass

    # build state dict from arrays in file (excluding 'meta')
    state = {k: data[k] for k in data.files if k != "meta"}

    # create model if needed
    if model is None:
        if model_ctor is None:
            raise ValueError("Provide either an existing `model` or a `model_ctor` to instantiate.")
        ctor_kwargs = dict(ctor_kwargs or {})
        # best-effort: fill ctor kwargs from meta if not provided
        for key in ("img_size", "num_classes", "lr"):
            if key in meta and key not in ctor_kwargs and meta[key] is not None:
                ctor_kwargs[key] = meta[key]
        model = model_ctor(**ctor_kwargs)

    report = load_state_dict(model, state, strict=strict)
    print(f"✅ Loaded model from {path} | tensors loaded: {len(report['loaded'])}")
    if report["missing_in_model"]:
        print(f"… skipped (missing in model): {report['missing_in_model']}")
    if report["missing_in_state"]:
        print(f"… missing in state (kept init): {report['missing_in_state']}")
    if report["shape_mismatch"]:
        print(f"… shape mismatch: {report['shape_mismatch']}")
    return model, meta, report


## Part C / Task 7,8 — **Your Model** (write code here)

Implement a small CNN for classification using **only** your Task 2 functions (`conv2d`, `ReLU`, `max_pool2d`, `linear_layer`, `backward`, `update_params`, etc.).

**Example topology: (Can be modified)**  

`Conv(1→f1, 5×5) → ReLU → MaxPool(2×2) → Conv(f1→f2, 5×5)* → ReLU → MaxPool(2×2) → Flatten → FC → ReLU → FC`  

\* Since `conv2d` here is single-channel, you can implement the second conv as “sum over channels” (loop over input channels) or keep **f1=1** to keep it simple.

**You must implement:**

- `forward(X)`: returns `logits` of shape `(N, num_classes)` **and** any caches required for backward.

- `backward(dlogits, cache)`: compute gradients for all parameters and update them using `update_params`.

**Constraints:** Use **only NumPy** and the functions from Section 1. **No PyTorch/TensorFlow**.

In `forward` we already implemented:

- **Block 1**: `Conv1(1→f1, k1) → ReLU → MaxPool2d`

- **Head**: `Flatten → FC1 → ReLU → FC2`

You must implement the **middle block**, here is an example, you can add more layer based on this:

- **Block 2**: `Conv2(f1→f2, k2) with sum over input channels → ReLU → MaxPool2d`


## Task 7: Finish the forward function part of Block 2

## Task 8: Finish the backward function part of Block 2

In [12]:
# ========== 快速卷积优化（保持不变） ==========
from numpy.lib.stride_tricks import as_strided
def conv2d_fast(X: np.ndarray, kernel: np.ndarray) -> np.ndarray:
    """
    快速向量化卷积实现 - 替换原来的循环实现
    """
    H, W = X.shape
    k = kernel.shape[0]
    H_out, W_out = H - k + 1, W - k + 1

    # 创建滑动窗口视图（零拷贝）
    shape = (H_out, W_out, k, k)
    strides = (X.strides[0], X.strides[1], X.strides[0], X.strides[1])
    windows = as_strided(X, shape=shape, strides=strides)

    # 向量化计算
    return np.tensordot(windows, kernel, axes=((2,3),(0,1)))

def setup_fast_convolutions():
    """
    启用快速卷积 - 在训练开始前调用此函数
    """
    global conv2d
    conv2d = conv2d_fast
    print("✅ 快速卷积已启用")

# 自动启用快速卷积
setup_fast_convolutions()

# ========== 改进的激活函数 ==========
def LeakyReLU(X: np.ndarray, alpha=0.01) -> np.ndarray:
    """Leaky ReLU - 避免死亡神经元"""
    return np.where(X > 0, X, alpha * X)

def LeakyReLU_backward(dY, X, alpha=0.01):
    """Leaky ReLU的反向传播"""
    return dY * np.where(X > 0, 1, alpha)

# ========== 改进的权重初始化 ==========
def he_init(shape):
    """He初始化 - 适合ReLU族激活函数"""
    if len(shape) == 4:  # 卷积核 (out_ch, k, k, in_ch)
        fan_in = shape[1] * shape[2] * shape[3]
    elif len(shape) == 2:  # 全连接层
        fan_in = shape[0]
    elif len(shape) == 3:  # 单通道卷积核 (filters, k, k)
        fan_in = shape[1] * shape[2]
    else:
        return np.zeros(shape)

    scale = np.sqrt(2.0 / fan_in)
    return np.random.randn(*shape).astype(np.float32) * scale

# ========== 改进的CNN模型 ==========
class SmallCNN:
    def __init__(self, img_size: int, num_classes: int, lr: float = 0.01,
                 f1: int = 8, f2: int = 16, k1: int = 5, k2: int = 5, fc_hidden: int = 64):
        self.img_size = img_size
        self.num_classes = num_classes
        self.lr = lr
        self.f1, self.f2 = f1, f2
        self.k1, self.k2 = k1, k2
        self.fc_hidden = fc_hidden

        # ----- 改进的参数初始化 -----
        self.K1 = he_init((f1, k1, k1))
        self.K2 = he_init((f2, k2, k2, f1))

        # 计算特征图尺寸
        s1 = img_size - k1 + 1
        s1p = s1 // 2
        s2 = s1p - k2 + 1
        s2p = s2 // 2
        self.flat_dim = s2p * s2p * f2

        # 全连接层
        self.W1 = he_init((self.flat_dim, fc_hidden))
        self.b1 = np.zeros((fc_hidden,), dtype=np.float32)
        self.W2 = he_init((fc_hidden, num_classes))
        self.b2 = np.zeros((num_classes,), dtype=np.float32)

        # 动量优化
        self.momentum = 0.9
        self.init_momentum_buffers()

        print(f"✅ ImprovedSmallCNN初始化完成")

    def init_momentum_buffers(self):
        """初始化动量缓冲区"""
        self.v_K1 = np.zeros_like(self.K1)
        self.v_K2 = np.zeros_like(self.K2)
        self.v_W1 = np.zeros_like(self.W1)
        self.v_W2 = np.zeros_like(self.W2)
        self.v_b1 = np.zeros_like(self.b1)
        self.v_b2 = np.zeros_like(self.b2)

    def forward(self, X: np.ndarray):
        """
        X: (N,H,W) grayscale in [0,1]
        Return: logits (N,C), cache
        """
        N, H, W = X.shape

        # -------- Block 1 (Conv1 → LeakyReLU → Pool) --------
        C1, cache1 = conv2d_multi_forward_basic(X, self.K1)
        A1 = LeakyReLU(C1, alpha=0.1)
        P1, mask1 = maxpool2d_forward_basic(A1, k=2)

        # -------- Block 2 --------
        C2, cache2 = conv2d_mimo_forward_basic(P1, self.K2)
        A2 = LeakyReLU(C2, alpha=0.1)
        P2, mask2 = maxpool2d_forward_basic(A2, k=2)

        # -------- Head (Flatten → FC1 → LeakyReLU → FC2) --------
        Flt = P2.reshape(N, -1)
        Z1 = linear_layer(Flt, self.W1, self.b1)
        A3 = LeakyReLU(Z1, alpha=0.1)
        Z2 = linear_layer(A3, self.W2, self.b2)

        cache = (X, C1, A1, P1, mask1, cache1, C2, A2, P2, mask2, cache2, Flt, Z1, A3, Z2)
        return Z2, cache

    def backward(self, dlogits, cache):
        """
        改进的backward实现，使用动量优化
        """
        (X, C1, A1, P1, mask1, cache1,
         C2, A2, P2, mask2, cache2,
         Flt, Z1, A3, Z2) = cache

        # -------- FC Head backward --------
        dW2, db2, dA3 = backward(dlogits, A3, self.W2)
        dZ1 = LeakyReLU_backward(dA3, Z1, alpha=0.1)
        dW1, db1, dFlt = backward(dZ1, Flt, self.W1)

        # Reshape gradient back to P2 shape
        dP2 = dFlt.reshape(P2.shape)

        # -------- Block 2 backward --------
        dA2 = maxpool2d_backward_basic(dP2, mask2, k=2)
        dC2 = LeakyReLU_backward(dA2, C2, alpha=0.1)
        dP1, dK2 = conv2d_mimo_backward_basic(dC2, cache2)

        # -------- Block 1 backward --------
        dA1 = maxpool2d_backward_basic(dP1, mask1, k=2)
        dC1 = LeakyReLU_backward(dA1, C1, alpha=0.1)
        dX, dK1 = conv2d_multi_backward_basic(dC1, cache1)

        # -------- 使用动量更新参数 --------
        self.update_with_momentum(dK1, dK2, dW1, dW2, db1, db2)

    def update_with_momentum(self, dK1, dK2, dW1, dW2, db1, db2):
        """动量优化更新"""
        # 更新卷积层参数
        self.v_K1 = self.momentum * self.v_K1 - self.lr * dK1
        self.v_K2 = self.momentum * self.v_K2 - self.lr * dK2
        self.K1 += self.v_K1
        self.K2 += self.v_K2

        # 更新全连接层参数
        self.v_W1 = self.momentum * self.v_W1 - self.lr * dW1
        self.v_W2 = self.momentum * self.v_W2 - self.lr * dW2
        self.v_b1 = self.momentum * self.v_b1 - self.lr * db1
        self.v_b2 = self.momentum * self.v_b2 - self.lr * db2
        self.W1 += self.v_W1
        self.W2 += self.v_W2
        self.b1 += self.v_b1
        self.b2 += self.v_b2



✅ 快速卷积已启用


## Part D — Training & Evaluation (Do not Modify)

Fill your model above, then run the following cell to train and evaluate.

This will print progress, track time, and report validation accuracy each epoch.

In [13]:
def accuracy(logits, y):
    preds = np.argmax(softmax(logits), axis=1)
    return float(np.mean(preds == y))

In [14]:
def train_eval(
    data_root='dataset_small',
    img_size=100,
    batch_size=32,
    epochs=10,
    lr=0.01,
    use_augment=True,
    model_cls=None,
    save_path="checkpoints/model_final.npz",
):
    # ---- Data ----
    train_items, classes = list_images(os.path.join(data_root, 'train'))
    val_items, _ = list_images(os.path.join(data_root, 'val'))
    num_classes = len(classes)
    print(f'Classes: {classes} (num_classes={num_classes})')

    # ---- Model ----
    if model_cls is None:
        model_cls = SmallCNN  # default
    model = model_cls(img_size=img_size, num_classes=num_classes, lr=lr)

    t0_total = time.time()
    last_train_loss, last_val_loss = None, None

    # ---- Prepare save path ----
    save_dir="checkpoints"
    os.makedirs(save_dir, exist_ok=True)
    timestamp = time.strftime("%Y%m%d_%H%M%S")
    save_name = f"{model_cls}_{os.path.basename(data_root)}_e{epochs}_{timestamp}.npz"
    save_path = os.path.join(save_dir, save_name)

    print(f"[info] Model will be saved to: {save_path}")

    for ep in range(1, epochs+1):
        print(f"\n=== Epoch {ep}/{epochs} ===")
        random.shuffle(train_items)

        # ---- Train ----
        t0 = time.time()
        loss_accum, acc_accum, seen = 0.0, 0.0, 0
        pbar = tqdm(range(0, len(train_items), batch_size), desc=f"Train {ep}", ncols=80)
        for s in pbar:
            X, y = load_batch(train_items, s, min(s+batch_size, len(train_items)), img_size, augment=use_augment)
            logits, cache = model.forward(X)
            probs = softmax(logits)
            loss, dlogits = cross_entropy_loss(probs, y)
            model.backward(dlogits, cache)

            acc = accuracy(logits, y)
            bsz = X.shape[0]
            loss_accum += loss * bsz
            acc_accum += acc * bsz
            seen += bsz
            pbar.set_postfix(loss=f"{loss:.3f}", acc=f"{acc*100:.1f}%")

        last_train_loss = loss_accum / max(1, seen)
        train_time = time.time() - t0
        print(f"Epoch {ep:02d} Train: loss={last_train_loss:.4f}, acc={acc_accum/seen:.3f}, time={train_time:.1f}s")

        # ---- Val ----
        t1 = time.time()
        val_loss, val_acc, vseen = 0.0, 0.0, 0
        pbar_val = tqdm(range(0, len(val_items), batch_size), desc=f"Val {ep}", ncols=80)
        for s in pbar_val:
            Xv, yv = load_batch(val_items, s, min(s+batch_size, len(val_items)), img_size, augment=False)
            logits, _ = model.forward(Xv)
            probs = softmax(logits)
            l, _ = cross_entropy_loss(probs, yv)
            a = accuracy(logits, yv)
            bsz = Xv.shape[0]
            val_loss += l * bsz
            val_acc += a * bsz
            vseen += bsz
            pbar_val.set_postfix(loss=f"{l:.3f}", acc=f"{a*100:.1f}%")

        last_val_loss = val_loss / max(1, vseen)
        val_time = time.time() - t1
        print(f"Epoch {ep:02d} Val  : loss={last_val_loss:.4f}, acc={val_acc/vseen:.3f}, time={val_time:.1f}s")

    total_time = time.time() - t0_total
    print(f"\n✅ Training finished in {total_time/60:.2f} minutes.")
    print(f"🔚 Final losses — train: {last_train_loss:.4f}, val: {last_val_loss:.4f}")

    # ---- Save once at the end (works for any model structure with ndarray params) ----
    if save_path:
        save_model(
            model,
            classes,
            path=save_path,
            extra_meta={
                "best_val_loss": float(last_val_loss),
                "final_train_loss": float(last_train_loss),
                "data_root": data_root,
            },
        )

    return model, classes, {"final_train_loss": last_train_loss, "final_val_loss": last_val_loss}


In [15]:
def run_test_submit(
    data_root='dataset_small',
    img_size=100,
    batch_size=64,
    load_path="checkpoints/model_final.npz",
    model_cls=None,
    strict=True,
):
    """
    Load a trained model and evaluate it on data_root/test.
    If test/ has subfolders (class names), computes accuracy.
    Otherwise just runs inference and reports number of samples.
    """
    # --- Load model ---
    if model_cls is None:
        model_cls = SmallCNN
    model, meta, report = load_model(
        load_path,
        model_ctor=model_cls,
        ctor_kwargs={"img_size": img_size},
        strict=strict,
    )

    # --- Load test items ---
    classes_meta = meta.get("classes", None)
    test_dir = os.path.join(data_root, "test")
    if not os.path.exists(test_dir):
        raise FileNotFoundError(f"Test directory not found: {test_dir}")

    # check if test has labeled subfolders (e.g., test/classA/...)
    if any(os.path.isdir(os.path.join(test_dir, d)) for d in os.listdir(test_dir)):
        test_items, classes_disk = list_images(test_dir)
        classes = classes_meta if classes_meta else classes_disk
        cls_to_idx = {c: i for i, c in enumerate(classes)}
        y_true = np.array([cls_to_idx[os.path.basename(os.path.dirname(p))] for p, _ in test_items], dtype=np.int32)
    else:
        # unlabeled: flat structure, only evaluate inference speed/count
        test_items = [(os.path.join(test_dir, f), -1) for f in os.listdir(test_dir)]
        classes = classes_meta if classes_meta else []
        y_true = None

    print(f"Loaded model from {load_path}")
    print(f"Test samples: {len(test_items)}, labeled={y_true is not None}")
    if classes:
        print(f"Classes ({len(classes)}): {classes}")

    # --- Inference ---
    logits_list = []
    for s in tqdm(range(0, len(test_items), batch_size), desc="Infer", ncols=80):
        X, _ = load_batch(test_items, s, min(s+batch_size, len(test_items)), img_size, augment=False)
        logits, _ = model.forward(X)
        logits_list.append(logits)

    logits = np.concatenate(logits_list, axis=0)
    probs = softmax(logits)
    preds = np.argmax(probs, axis=1)

    # --- Accuracy if ground truth available ---
    if y_true is not None and len(y_true) == len(preds):
        acc = float((preds == y_true).mean())
        print(f"📊 Test accuracy: {acc:.4f}")
    else:
        acc = None
        print("📊 Test set appears unlabeled — accuracy not computed.")

    return acc


## Part E — Run Training

You can tune the following hyperparameters:
- Hyperparameters

    - `batch_size` (e.g., 16 / 32 / 64)

    - `epochs` (e.g., 5 / 10 / 20)

    - `lr`: learning rate (e.g., 0.001 / 0.01 / 0.05)

    - `use_augment` (True / False)

**Tip:** Start with small epochs to make sure your implementation runs end-to-end.


In [16]:
drive.mount('/content/drive')


Mounted at /content/drive


In [17]:
def main(
    mode="train",
    data_root=  '/content/drive/MyDrive/dataset',
    img_size=100,
    batch_size=32,
    epochs=25,
    lr=0.01,
    use_augment=True,
    model_cls=None,                        # SmallCNN
    save_path="checkpoints/model_final.npz",
    load_path=None,                        # if None, defaults to save_path
    strict=False,
    submit_batch_size=64,
    save_csv="submission.csv",
):
    """
    mode='train'  -> trains, saves model.
    mode='test' -> loads saved model and runs test inference.
    """
    if mode == "train":
        return train_eval(
            data_root=data_root,
            img_size=img_size,
            batch_size=batch_size,
            epochs=epochs,
            lr=lr,
            use_augment=use_augment,
            model_cls=model_cls,
            save_path=save_path,
        )
    elif mode == "test":
        lp = load_path or save_path
        return run_test_submit(
            data_root=data_root,
            img_size=img_size,
            batch_size=submit_batch_size,
            load_path=lp,
            model_cls=model_cls,
            strict=strict,
        )
    else:
        raise ValueError("mode must be 'train' or 'test'")


In [19]:
# Example run
# NOTE: You must finish Parts A and C before running.
# Train and save a SmallCNN

main(mode="train",                        # set "test" when submitting your bouns task
    data_root= '/content/drive/MyDrive/dataset',
    img_size=100,
    batch_size=32,
    epochs=25,
    lr=0.01,
    use_augment=True,
    model_cls=SmallCNN,
    save_path="checkpoints/model_final.npz",
    load_path=None,
    strict=True,
    submit_batch_size=64,
)


Classes: ['bird', 'car', 'cat', 'dog', 'person'] (num_classes=5)
✅ ImprovedSmallCNN初始化完成
[info] Model will be saved to: checkpoints/<class '__main__.SmallCNN'>_dataset_e25_20251121_165245.npz

=== Epoch 1/25 ===


Train 1: 100%|███████████| 16/16 [05:06<00:00, 19.18s/it, acc=35.0%, loss=1.613]


Epoch 01 Train: loss=1.6396, acc=0.170, time=306.9s


Val 1: 100%|████████████████| 8/8 [02:22<00:00, 17.79s/it, acc=7.7%, loss=1.594]


Epoch 01 Val  : loss=1.6026, acc=0.232, time=142.3s

=== Epoch 2/25 ===


...

Train 25: 100%|█████████| 16/16 [04:01<00:00, 15.10s/it, acc=100.0%, loss=0.055]


Epoch 25 Train: loss=0.1877, acc=0.942, time=241.6s


Val 25: 100%|██████████████| 8/8 [01:29<00:00, 11.20s/it, acc=46.2%, loss=2.579]

Epoch 25 Val  : loss=3.5360, acc=0.304, time=89.6s

✅ Training finished in 140.41 minutes.
🔚 Final losses — train: 0.1877, val: 3.5360
💾 Saved 12 tensors to checkpoints/<class '__main__.SmallCNN'>_dataset_e25_20251121_165245.npz


(<__main__.SmallCNN at 0x7fe848157110>,
 ['bird', 'car', 'cat', 'dog', 'person'],
 {'final_train_loss': 0.18769719198346138,
  'final_val_loss': 3.5360076236724853})

In [20]:
# 训练完成后运行这个
import shutil
import os

# 找到最新生成的模型文件
checkpoint_dir = "checkpoints"
files = os.listdir(checkpoint_dir)
latest_file = sorted(files)[-1]  # 按文件名排序，取最新的

# 复制到你的固定路径
source_path = os.path.join(checkpoint_dir, latest_file)
target_path = "/content/drive/MyDrive/dataset/model_final.npz"

shutil.copy(source_path, target_path)
print(f"✅ 已复制到固定路径: {target_path}")

# 下载到本地
from google.colab import files
files.download(target_path)

✅ 已复制到固定路径: /content/drive/MyDrive/dataset/model_final.npz


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>